In [1]:
import numpy as np
import pandas as pd

# 영화 예시 평점 행렬 R (0은 아직 평점이 없는 항목)
data = {
    '사용자': ['사용자 A', '사용자 B', '사용자 C', '사용자 D'],
    '기생충': [5, 4, 1, 0],
    '부산행': [0, 2, 0, 4],
    '도둑들': [3, 0, 5, 4],
    '태극기휘날리며': [1, 0, 4, 2]
}

# 데이터프레임으로 변환
df = pd.DataFrame(data)
df.set_index('사용자', inplace=True)

print('평점 행렬 R:')
print(df.shape)
print(df)


평점 행렬 R:
(4, 4)
       기생충  부산행  도둑들  태극기휘날리며
사용자                          
사용자 A    5    0    3        1
사용자 B    4    2    0        0
사용자 C    1    0    5        4
사용자 D    0    4    4        2


In [2]:
# 행렬 분해 함수
def matrix_factorization(R,             # 사용자-아이템 평점 행렬
                         K,             # 잠재 요인 개수(k=2, k=6)
                         steps=5000,    # SGD 학습 반복 횟수 5000
                         alpha=0.0002,  # 학습률(learning Rate, 0.0002)
                         beta=0.02):    # 정규화 계수
    N = len(R)      # 사용자의 수
    M = len(R[0])   # 아이템(영화) 개수

    # 사용자 잠재 요인 행렬 P (N x K)
    P = np.random.rand(N, K)

    # 아이템 잠재 요인 행렬 Q (M x K)
    Q = np.random.rand(M, K)

    # Q.T를 구함
    Q = Q.T

    # 학습 과정
    for step in range(steps):
        for i in range(N):
            for j in range(M):
                if R[i][j] > 0:
                    # 예측된 평점
                    eij = R[i][j] - np.dot(P[i, :], Q[:, j])    # 실제 평점과, 예측 평점 간의 오차

                    # 경사 하강법으로 P와 Q 업데이트
                    for k in range(K):
                        P[i][k] = P[i][k] + alpha * (2 * eij * Q[k][j] - beta * P[i][k])    # -> 오차를 기반으로 P와 Q를 보정
                        Q[k][j] = Q[k][j] + alpha * (2 * eij * P[i][k] - beta * Q[k][j])

        # 손실 함수 계산
        eR = np.dot(P, Q)
        e = 0
        for i in range(N):
            for j in range(M):
                if R[i][j] > 0:
                    e = e + pow(R[i][j] - np.dot(P[i, :], Q[:, j]), 2)
                    for k in range(K):
                        e = e + (beta/2) * (pow(P[i][k], 2) + pow(Q[k][j], 2))
        if e < 0.001:   # 오차가 0.001보다 작아지면 학습 조기 종료
            break

    return P, Q.T       # 최종 분해된 P, Q 행렬 반환

In [3]:
# 평점 행렬을 넘파이 배열로 변환
R = df.values

# 잠재 요인 수 K
K = 2

# 행렬 분해 수행
P, Q = matrix_factorization(R, K)

# 예측 평점 행렬
nR = np.dot(P, Q.T)

# 결과를 데이터프레임으로 변환
nR_df = pd.DataFrame(nR, index=df.index, columns=df.columns)

print('원래 평점 행렬 R:')
print(df)
print('-'*50)

print('\n 사용자 잠재 요인 행렬 P:')
print(pd.DataFrame(P, index=df.index))
print('-'*50)

print('\n 아이템 잠재 요인 행렬 Q:')
print(pd.DataFrame(Q, index=df.columns))
print('-'*50)

print('\n 예측 평점 행렬 nR:')      # 사용자가 평가하지 않은 0 값이 예측 평점으로 채워짐 -> 추천 가능
print(nR_df)

원래 평점 행렬 R:
       기생충  부산행  도둑들  태극기휘날리며
사용자                          
사용자 A    5    0    3        1
사용자 B    4    2    0        0
사용자 C    1    0    5        4
사용자 D    0    4    4        2
--------------------------------------------------

 사용자 잠재 요인 행렬 P:
              0         1
사용자                      
사용자 A -0.204145  2.061461
사용자 B  0.527839  1.739683
사용자 C  2.435309  0.724694
사용자 D  1.590827  0.710580
--------------------------------------------------

 아이템 잠재 요인 행렬 Q:
                0         1
기생충     -0.304989  2.384650
부산행      2.222880  0.489141
도둑들      1.637093  1.622537
태극기휘날리며  1.319158  0.595534
--------------------------------------------------

 예측 평점 행렬 nR:
            기생충       부산행       도둑들   태극기휘날리며
사용자                                          
사용자 A  4.978126  0.554555  3.010592  0.958371
사용자 B  3.987550  2.024273  3.686821  1.732343
사용자 C  0.985400  5.767877  5.162671  3.644137
사용자 D  1.209299  3.883792  3.757274  2.521727


In [4]:
# 더 많은 데이터의 경우
data = {
    '사용자': ['사용자 A', '사용자 B', '사용자 C', '사용자 D', '사용자 E'],
    '기생충': [5, 4, 1, 0, 3],
    '부산행': [0, 2, 0, 4, 0],
    '태극기 휘날리며': [3, 0, 5, 4, 0],
    '도둑들': [1, 0, 4, 2, 4],
    '설국열차': [0, 3, 0, 0, 5],
    '범죄도시': [2, 0, 4, 0, 3],
}

# 데이터프레임으로 변환
df = pd.DataFrame(data)
df.set_index('사용자', inplace=True)

# 행렬 분해 함수
def matrix_factorization(R, K, steps=5000, alpha=0.0002, beta=0.02):
    N = len(R)
    M = len(R[0])

    # 사용자 잠재 요인 행렬 P (N x K)
    P = np.random.rand(N, K)

    # 아이템 잠재 요인 행렬 Q (M x K)
    Q = np.random.rand(M, K)

    # Q.T를 구함
    Q = Q.T

    # 학습 과정
    for step in range(steps):
        for i in range(N):
            for j in range(M):
                if R[i][j] > 0:
                    # 예측된 평점
                    eij = R[i][j] - np.dot(P[i, :], Q[:, j])

                    # 경사 하강법으로 P와 Q 업데이트
                    for k in range(K):
                        P[i][k] = P[i][k] + alpha * (2 * eij * Q[k][j] - beta * P[i][k])
                        Q[k][j] = Q[k][j] + alpha * (2 * eij * P[i][k] - beta * Q[k][j])

        # 손실 함수 계산
        eR = np.dot(P, Q)
        e = 0
        for i in range(N):
            for j in range(M):
                if R[i][j] > 0:
                    e = e + pow(R[i][j] - np.dot(P[i, :], Q[:, j]), 2)
                    for k in range(K):
                        e = e + (beta/2) * (pow(P[i][k], 2) + pow(Q[k][j], 2))
        if e < 0.001:
            break

    return P, Q.T

# 평점 행렬을 넘파이 배열로 변환
R = df.values

# 잠재 요인 수 K
K = 2

# 행렬 분해 수행
P, Q = matrix_factorization(R, K)

# 예측 평점 행렬
nR = np.dot(P, Q.T)

# 결과를 데이터프레임으로 변환
nR_df = pd.DataFrame(nR, index=df.index, columns=df.columns)

In [5]:
print('원래 평점 행렬 R:')
print(df)
print('-'*50)

print('\n 사용자 잠재 요인 행렬 P:')
print(pd.DataFrame(P, index=df.index, columns=['잠재요인1', '잠재요인2']))
print('-'*50)

print('\n 아이템 잠재 요인 행렬 Q:')
print(pd.DataFrame(Q, index=df.columns, columns=['잠재요인1', '잠재요인2']))
print('-'*50)

print('\n 예측 평점 행렬 nR:')      # 사용자가 평가하지 않은 0 값이 예측 평점으로 채워짐 -> 추천 가능
print(nR_df)

원래 평점 행렬 R:
       기생충  부산행  태극기 휘날리며  도둑들  설국열차  범죄도시
사용자                                       
사용자 A    5    0         3    1     0     2
사용자 B    4    2         0    0     3     0
사용자 C    1    0         5    4     0     4
사용자 D    0    4         4    2     0     0
사용자 E    3    0         0    4     5     3
--------------------------------------------------

 사용자 잠재 요인 행렬 P:
          잠재요인1     잠재요인2
사용자                      
사용자 A  0.799808  1.868548
사용자 B  0.973552  1.304987
사용자 C  2.558165 -0.132467
사용자 D  1.392372  1.887300
사용자 E  2.209652  0.667949
--------------------------------------------------

 아이템 잠재 요인 행렬 Q:
             잠재요인1     잠재요인2
기생충       0.568344  2.470040
부산행       1.090354  1.124416
태극기 휘날리며  1.964429  0.730835
도둑들       1.661554 -0.125387
설국열차      2.090281  0.618123
범죄도시      1.442715  0.373833
--------------------------------------------------

 예측 평점 행렬 nR:
            기생충       부산행  태극기 휘날리며       도둑들      설국열차      범죄도시
사용자                             